## NBDev Default Export
Set the default export module name for nbdev.

In [ ]:
#| default_exp memorytools

## NBDev Hide
Import necessary nbdev functions like showdoc.

In [ ]:
#| hide
from nbdev.showdoc import *

## Import Required Libraries
Import `SemanticMemory` from the memory module, `Chat`, `toolloop`, `models` from `claudette`, and other necessary libraries like `json`.

In [ ]:
#| export
import json
from cogitarelink.memory import SemanticMemory # Assuming 00_memory.ipynb exports SemanticMemory
from cogitarelink.retriever import LODNavigator
import json
from pyld import jsonld
import httpx
from rdflib import Graph, Dataset, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from datetime import datetime
from enum import Enum
from typing import List, Literal, Optional, Dict, Any
from pydantic import BaseModel, Field
from fastcore.basics import patch
from dataclasses import dataclass, field
from claudette import Chat, toolloop, models
import time
from pathlib import Path
import pickle

/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Example of Semantic Memory Manager Agent

### Here's the list of functions that should go into the memTools module:

1. **analyze_vocabulary_structure**
   - Analyzes the structure of a vocabulary (classes, properties, hierarchy)
   - Returns vocabulary metadata, classes, properties, and relationships

2. **find_vocabulary_term**
   - Searches for terms across vocabularies based on a search string
   - Returns matching classes and properties with their details

3. **list_entities**
   - Lists all entities in the memory's by_id index
   - Returns a list of entity IDs

4. **get_entity**
   - Gets details about a specific entity by ID
   - Returns simplified entity data with properties and values

5. **find_entity_relationships**
   - Finds relationships to/from a specific entity
   - Returns incoming and outgoing relationships

6. **semantic_memory_agent**
   - Agent that can work with all semantic memory components
   - Provides tools for exploring vocabularies, contexts, and instance data
   - Answers questions about the semantic data

7. **debug functions**
   - debug_vocabulary_loading
   - debug_semantic_agent_tools
   - debug_tool_execution
   - test_semantic_reasoning_simple
   - test_semantic_reasoning_extended
   - test_vocabulary_management
   - test_vocabulary_agent
   - test_enhanced_semantic_agent
   - test_complex_semantic_integration

These functions build upon the core functionality of the semantic memory system rather than modifying it, which is why they belong in the memTools module.

In [ ]:
#| export
def analyze_vocabulary_structure(memory: SemanticMemory, vocab_uri: str) -> dict:
    """Analyze the structure of a vocabulary in memory
    
    Args:
        memory: The semantic memory instance
        vocab_uri: URI of the vocabulary to analyze
        
    Returns:
        Dictionary with vocabulary structure information
    """
    # Check if vocabulary exists in registry
    if vocab_uri not in memory.resource_registry["vocabularies"]:
        return {"error": f"Vocabulary {vocab_uri} not found in memory"}
    
    # Collect classes
    classes = []
    for s, p, o in memory.default_graph.triples((None, RDF.type, RDFS.Class)):
        if str(s).startswith(vocab_uri):
            class_info = {
                "uri": str(s),
                "name": str(s).split("/")[-1]
            }
            
            # Get label if available
            for _, _, label in memory.default_graph.triples((s, RDFS.label, None)):
                class_info["label"] = str(label)
                break
                
            # Get comment if available
            for _, _, comment in memory.default_graph.triples((s, RDFS.comment, None)):
                class_info["description"] = str(comment)
                break
                
            classes.append(class_info)
    
    # Collect properties
    properties = []
    for s, p, o in memory.default_graph.triples((None, RDF.type, RDF.Property)):
        if str(s).startswith(vocab_uri):
            prop_info = {
                "uri": str(s),
                "name": str(s).split("/")[-1]
            }
            
            # Get label if available
            for _, _, label in memory.default_graph.triples((s, RDFS.label, None)):
                prop_info["label"] = str(label)
                break
                
            # Get domain and range if available
            for _, _, domain in memory.default_graph.triples((s, RDFS.domain, None)):
                prop_info["domain"] = str(domain)
            for _, _, range_val in memory.default_graph.triples((s, RDFS.range, None)):
                prop_info["range"] = str(range_val)
                
            properties.append(prop_info)
    
    # Collect class hierarchy
    hierarchy = []
    for s, p, o in memory.default_graph.triples((None, RDFS.subClassOf, None)):
        if str(s).startswith(vocab_uri) or str(o).startswith(vocab_uri):
            hierarchy.append({
                "subclass": str(s),
                "superclass": str(o)
            })
    
    # Get metadata from registry
    metadata = memory.resource_registry["vocabularies"].get(vocab_uri, {})
    
    return {
        "uri": vocab_uri,
        "description": metadata.get("description", "No description"),
        "last_updated": metadata.get("cached_at", 0),
        "class_count": len(classes),
        "property_count": len(properties),
        "hierarchy_count": len(hierarchy),
        "classes": classes[:20],  # Limit to avoid overwhelming context
        "properties": properties[:20],
        "hierarchy": hierarchy[:20]
    }

In [ ]:
#| export
def find_vocabulary_term(memory: SemanticMemory, search_term: str) -> list:
    """Search for terms across vocabularies
    
    Args:
        memory: The semantic memory instance
        search_term: Term to search for
        
    Returns:
        List of matching terms with their details
    """
    results = []
    
    # Search for classes with matching names or labels
    for s, p, o in memory.default_graph.triples((None, RDF.type, RDFS.Class)):
        term_name = str(s).split("/")[-1].lower()
        if search_term.lower() in term_name:
            term_info = {
                "uri": str(s),
                "name": term_name,
                "type": "class",
                "vocabulary": str(s).rsplit("/", 1)[0] + "/"
            }
            
            # Get label if available
            for _, _, label in memory.default_graph.triples((s, RDFS.label, None)):
                term_info["label"] = str(label)
                break
                
            results.append(term_info)
    
    # Also search for classes by label
    for s, p, o in memory.default_graph.triples((None, RDFS.label, None)):
        if search_term.lower() in str(o).lower():
            # Check if it's a class
            is_class = False
            for _, _, _ in memory.default_graph.triples((s, RDF.type, RDFS.Class)):
                is_class = True
                break
                
            if is_class:
                term_info = {
                    "uri": str(s),
                    "name": str(s).split("/")[-1],
                    "type": "class",
                    "label": str(o),
                    "vocabulary": str(s).rsplit("/", 1)[0] + "/"
                }
                results.append(term_info)
    
    # Search for properties with matching names or labels
    for s, p, o in memory.default_graph.triples((None, RDF.type, RDF.Property)):
        term_name = str(s).split("/")[-1].lower()
        if search_term.lower() in term_name:
            term_info = {
                "uri": str(s),
                "name": term_name,
                "type": "property",
                "vocabulary": str(s).rsplit("/", 1)[0] + "/"
            }
            
            # Get label if available
            for _, _, label in memory.default_graph.triples((s, RDFS.label, None)):
                term_info["label"] = str(label)
                break
                
            results.append(term_info)
    
    # Also search for properties by label
    for s, p, o in memory.default_graph.triples((None, RDFS.label, None)):
        if search_term.lower() in str(o).lower():
            # Check if it's a property
            is_property = False
            for _, _, _ in memory.default_graph.triples((s, RDF.type, RDF.Property)):
                is_property = True
                break
                
            if is_property:
                term_info = {
                    "uri": str(s),
                    "name": str(s).split("/")[-1],
                    "type": "property",
                    "label": str(o),
                    "vocabulary": str(s).rsplit("/", 1)[0] + "/"
                }
                results.append(term_info)
    
    return results

In [ ]:
#| export
def list_entities(memory: SemanticMemory) -> list:
    """List all entities in the by_id index
    
    Args:
        memory: The semantic memory instance
        
    Returns:
        List of entity IDs
    """
    entities = []
    for entity_id in memory.indices["by_id"].keys():
        entities.append(entity_id)
    return entities[:20]  # Limit to avoid overwhelming context

In [ ]:
#| export
def get_entity(memory: SemanticMemory, entity_id: str) -> dict:
    """Get a specific entity by ID with simplified output
    
    Args:
        memory: The semantic memory instance
        entity_id: ID of the entity to retrieve
        
    Returns:
        Simplified entity data
    """
    entity = memory.query_by_id(entity_id)
    if not entity:
        return {"error": f"Entity {entity_id} not found"}
        
    # Simplify the output for easier consumption by LLM
    result = {
        "id": entity.get("@id", "unknown"),
        "type": entity.get("@type", [])
    }
    
    # Extract properties with simpler formatting
    for key, value in entity.items():
        if key not in ["@id", "@type"]:
            # Simplify property name by removing URI prefix
            prop_name = key.split("/")[-1]
            
            # Simplify values
            simple_values = []
            for val in value:
                if isinstance(val, dict):
                    if "@value" in val:
                        simple_values.append(val["@value"])
                    elif "@id" in val:
                        simple_values.append(val["@id"])
                else:
                    simple_values.append(val)
                    
            result[prop_name] = simple_values
            
    return result

In [ ]:
#| export
def find_entity_relationships(memory: SemanticMemory, entity_id: str) -> dict:
    """Find all relationships to/from a specific entity
    
    Args:
        memory: The semantic memory instance
        entity_id: ID of the entity to analyze
        
    Returns:
        Dictionary of incoming and outgoing relationships
    """
    incoming = []
    outgoing = []
    
    # Find all triples where this entity is the subject
    for _, p, o in memory.default_graph.triples((URIRef(entity_id), None, None)):
        if isinstance(o, URIRef) and str(p) != str(RDF.type):
            outgoing.append({
                "predicate": str(p),
                "predicate_name": str(p).split("/")[-1],
                "object": str(o)
            })
    
    # Find all triples where this entity is the object
    for s, p, _ in memory.default_graph.triples((None, None, URIRef(entity_id))):
        if str(p) != str(RDF.type):
            incoming.append({
                "subject": str(s),
                "predicate": str(p),
                "predicate_name": str(p).split("/")[-1]
            })
    
    return {
        "entity": entity_id,
        "outgoing_relationships": outgoing,
        "incoming_relationships": incoming
    }

### Core Semantic Memory Agent

In [ ]:
#| export
def semantic_memory_agent(memory: SemanticMemory, navigator, query: str, model: str = "claude-3-7-sonnet-20250219"):
    """Enhanced agent that can work with all semantic memory components: contexts, vocabularies, and instance data
    
    Args:
        memory: SemanticMemory instance
        navigator: LODNavigator instance
        query: User query about semantic memory
        model: Claude model to use
    
    Returns:
        Claude's response
    """
    # PART 1: VOCABULARY TOOLS
    def get_vocabularies() -> list:
        """List all vocabularies in memory"""
        vocabs = []
        for uri, metadata in memory.resource_registry["vocabularies"].items():
            vocabs.append({
                "uri": uri,
                "description": metadata.get("description", "No description"),
                "last_updated": metadata.get("cached_at", 0)
            })
        return vocabs
    
    def analyze_vocab(vocab_uri: str) -> dict:
        """Analyze the structure of a vocabulary"""
        return analyze_vocabulary_structure(memory, vocab_uri)
    
    def search_vocab_term(search_term: str) -> list:
        """Search for a term across vocabularies"""
        return find_vocabulary_term(memory, search_term)
    
    def load_vocab(vocab_name: str) -> dict:
        """Load a standard vocabulary if not already in memory"""
        success, data = memory.ensure_standard_vocabulary(vocab_name, navigator)
        return {
            "success": success,
            "vocab_name": vocab_name,
            "summary": "Vocabulary loaded successfully" if success else "Failed to load vocabulary"
        }
    
    # PART 2: CONTEXT TOOLS
    def get_contexts() -> list:
        """List all contexts in memory"""
        contexts = []
        for uri, metadata in memory.resource_registry["contexts"].items():
            contexts.append({
                "uri": uri,
                "description": metadata.get("description", "No description"),
                "last_updated": metadata.get("cached_at", 0)
            })
        return contexts
    
    # PART 3: INSTANCE DATA TOOLS
    def get_all_entities() -> list:
        """List all entities in memory"""
        return list_entities(memory)  # Now calling the global function
    
    def get_entity_details(entity_id: str) -> dict:
        """Get details about a specific entity"""
        return get_entity(memory, entity_id)  # Calling the global function
    
    def get_entity_relationships(entity_id: str) -> dict:
        """Find relationships for an entity"""
        return find_entity_relationships(memory, entity_id)  # Calling the global function
    
    def find_entities_by_type(type_uri: str) -> list:
        """Search for entities of a specific type"""
        entities = []
        for entity_id in memory.indices["by_id"].keys():
            entity = memory.query_by_id(entity_id)
            if entity and "@type" in entity:
                entity_types = entity["@type"] if isinstance(entity["@type"], list) else [entity["@type"]]
                if type_uri in entity_types:
                    entities.append(entity_id)
        return entities
    
    # Create a dictionary of all tools
    tools = {
        # Vocabulary tools
        "get_vocabularies": get_vocabularies,
        "analyze_vocab": analyze_vocab,
        "search_vocab_term": search_vocab_term,
        "load_vocab": load_vocab,
        
        # Context tools
        "get_contexts": get_contexts,
        
        # Instance data tools
        "get_all_entities": get_all_entities,
        "get_entity_details": get_entity_details,
        "get_entity_relationships": get_entity_relationships,
        "find_entities_by_type": find_entities_by_type
    }
    
    # Create a Claude chat with these tools
    try:
        chat = Chat(model=model, tools=list(tools.values()))
        
        # Create the prompt
        prompt = f"""
        You are a Semantic Knowledge Agent that helps users explore and understand all components of semantic memory:
        1. Contexts - JSON-LD contexts that map terms to URIs
        2. Vocabularies - Ontologies and schemas that define classes and properties
        3. Instance Data - Actual entities and their relationships
        
        You have access to these tools for working with vocabularies:
        - get_vocabularies: List all vocabularies in memory
        - analyze_vocab: Analyze the structure of a vocabulary (pass the full URI)
        - search_vocab_term: Search for a term across vocabularies
        - load_vocab: Load a standard vocabulary
        
        Tools for working with contexts:
        - get_contexts: List all contexts in memory
        
        Tools for working with instance data:
        - get_all_entities: List all entities in memory
        - get_entity_details: Get details about a specific entity
        - get_entity_relationships: Find relationships for an entity
        - find_entities_by_type: Search for entities of a specific type
        
        When answering questions about relationships between entities, first use get_all_entities to find available entities,
        then use get_entity_details to examine their details, and get_entity_relationships to see how they're connected.
        
        User question: {query}
        """
        
        # Use toolloop to let Claude use the tools
        response = chat.toolloop(prompt)
        return response
    except Exception as e:
        print(f"Error in semantic_memory_agent: {e}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"

In [ ]:
import dspy # Add dspy import

# Configure DSPy LM for this notebook's context
try:
    # Use the same model as in 01_retriever.ipynb
    lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
    dspy.configure(lm=lm)
    print("DSPy configured successfully.")
except Exception as e:
    print(f"Error configuring DSPy: {e}")
    # Optionally raise the error or handle it if configuration is critical
    # raise e

DSPy configured successfully.


In [ ]:
#| export
def test_enhanced_semantic_agent():
    """Test the enhanced semantic agent with all components"""
    print("\n=== Testing Enhanced Semantic Agent ===\n")
    
    # Reuse our test setup from before
    cache_dir = Path("./cache")
    memory = SemanticMemory(cache_dir=cache_dir)
    navigator = LODNavigator()
    
    print("Initializing test environment...")
    
    # Load at least one vocabulary
    print("\nLoading Dublin Core vocabulary:")
    memory.ensure_standard_vocabulary("dc", navigator)
    
    # Add instance data
    print("\nAdding instance data:")
    instance_data = [
        {
            "@context": {"@vocab": "http://schema.org/"},
            "@id": "http://example.org/person/alice",
            "@type": "Person",
            "name": "Alice Smith",
            "birthDate": "1990-01-01",
            "knows": {"@id": "http://example.org/person/bob"}
        },
        {
            "@context": {"@vocab": "http://schema.org/"},
            "@id": "http://example.org/person/bob",
            "@type": "Person",
            "name": "Bob Jones",
            "birthDate": "1985-05-15",
            "knows": {"@id": "http://example.org/person/alice"}
        }
    ]
    
    for instance in instance_data:
        memory.add_jsonld(instance)
    
    print(f"Added {len(instance_data)} instances to memory")
    
    # Test query that requires understanding instance data
    query = "How are Alice and Bob related to each other based on the instance data, and what vocabulary terms are used to describe their relationship?"
    
    print(f"\nTesting integrated query with enhanced agent: {query}")
    
    response = semantic_memory_agent(memory, navigator, query)
    
    print("\n=== Enhanced Agent Response ===\n")
    print(response)
    
    return memory, navigator, response

In [ ]:
mem, nav, res = test_enhanced_semantic_agent()


=== Testing Enhanced Semantic Agent ===

Initializing test environment...

Loading Dublin Core vocabulary:
Parsed JSON-LD graph with 99 entries
Added 700 triples from vocabulary http://purl.org/dc/terms/
Successfully loaded DC vocabulary with 700 triples

Adding instance data:
Added 2 instances to memory

Testing integrated query with enhanced agent: How are Alice and Bob related to each other based on the instance data, and what vocabulary terms are used to describe their relationship?

=== Enhanced Agent Response ===

Message(id='msg_01WqV6aGybTJD3XtcQJPkBgY', content=[TextBlock(citations=None, text='Based on the information gathered, here\'s what I can tell you about Alice and Bob\'s relationship:\n\n### Relationship between Alice and Bob\n\nAlice and Bob have a bidirectional "knows" relationship with each other:\n- Alice knows Bob (Alice has an outgoing relationship with predicate "knows" to Bob)\n- Bob knows Alice (Bob has an outgoing relationship with predicate "knows" to Alice)

In [ ]:
#| export
def debug_tool_execution():
    """Debug the tool execution process"""
    print("\n=== Debugging Tool Execution ===\n")
    
    # Create a simple memory and navigator
    memory = SemanticMemory()
    navigator = LODNavigator()
    
    # Load a vocabulary
    print("Loading vocabulary...")
    memory.ensure_standard_vocabulary("sosa", navigator)
    
    # Create a simple tool
    def get_vocab_info(vocab_uri: str) -> dict:
        """Get info about a vocabulary"""
        print(f"Tool called with URI: {vocab_uri}")
        return {"uri": vocab_uri, "status": "success"}
    
    # Create a chat with this tool
    tools = {"get_vocab_info": get_vocab_info}
    
    try:
        print("Creating chat with tool...")
        chat = Chat(model="claude-3-7-sonnet-20250219", tools=[get_vocab_info])
        
        # Create a simple prompt
        prompt = """
        You have access to a tool:
        - get_vocab_info: Get info about a vocabulary (pass the URI)
        
        Please get info about the SOSA vocabulary (http://www.w3.org/ns/sosa/).
        """
        
        print("Executing toolloop...")
        response = chat.toolloop(prompt, trace_func=print)
        
        print("\nResponse:")
        print(response)
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
    
    return "Debug complete"

In [ ]:
#| export
def debug_semantic_agent_tools():
    """Debug the semantic agent tools"""
    print("\n=== Debugging Semantic Agent Tools ===\n")
    
    # Create a simple memory and navigator
    memory = SemanticMemory()
    navigator = LODNavigator()
    
    # Create a test instance of the agent function
    def test_agent_function():
        # This will just create the tools and print their names
        # PART 1: VOCABULARY TOOLS
        def get_vocabularies() -> list:
            """List all vocabularies in memory"""
            return []
        
        def analyze_vocab(vocab_uri: str) -> dict:
            """Analyze the structure of a vocabulary"""
            return {}
        
        # Create a dictionary of all tools
        tools = {
            "get_vocabularies": get_vocabularies,
            "analyze_vocab": analyze_vocab,
        }
        
        # Print the tool names
        print("Available tool names:")
        for tool_name in tools.keys():
            print(f"- {tool_name}")
            
        # Check what the agent would see in the prompt
        prompt_snippet = """
        You have access to these tools for working with vocabularies:
        - get_vocabularies: List all vocabularies in memory
        - analyze_vocab: Analyze the structure of a vocabulary (pass the full URI)
        """
        print("\nPrompt snippet describing tools:")
        print(prompt_snippet)
        
        return tools
    
    # Run the test function
    tools = test_agent_function()
    
    return tools

In [ ]:
debug_semantic_agent_tools()


=== Debugging Semantic Agent Tools ===

Available tool names:
- get_vocabularies
- analyze_vocab

Prompt snippet describing tools:

        You have access to these tools for working with vocabularies:
        - get_vocabularies: List all vocabularies in memory
        - analyze_vocab: Analyze the structure of a vocabulary (pass the full URI)
        


{'get_vocabularies': <function __main__.debug_semantic_agent_tools.<locals>.test_agent_function.<locals>.get_vocabularies() -> list>,
 'analyze_vocab': <function __main__.debug_semantic_agent_tools.<locals>.test_agent_function.<locals>.analyze_vocab(vocab_uri: str) -> dict>}

In [ ]:
#| export
def test_simplified_agent():
    """Test the semantic agent with a simplified query"""
    print("\n=== Testing Simplified Semantic Agent ===\n")
    
    # Setup
    cache_dir = Path("./cache")
    memory = SemanticMemory(cache_dir=cache_dir)
    navigator = LODNavigator()
    
    print("Initializing test environment...")
    
    # Load just one vocabulary
    print("\nLoading SOSA vocabulary:")
    memory.ensure_standard_vocabulary("sosa", navigator)
    
    # Add a simple instance
    instance = {
        "@context": {
            "sosa": "http://www.w3.org/ns/sosa/",
            "Sensor": "sosa:Sensor",
            "Observation": "sosa:Observation",
            "madeBySensor": "sosa:madeBySensor",
            "observedProperty": "sosa:observedProperty",
            "hasResult": "sosa:hasResult"
        },
        "@id": "http://example.org/sensor/temp1",
        "@type": "Sensor"
    }
    
    memory.add_jsonld(instance)
    print("Added test sensor instance")
    
    # Simple query
    query = "What vocabularies are available in the system?"
    
    print(f"\nTesting simple query: {query}")
    
    try:
        response = semantic_memory_agent(memory, navigator, query)
        print("\n=== Agent Response ===\n")
        print(response)
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
    
    return memory, navigator

In [ ]:
test_simplified_agent()


=== Testing Simplified Semantic Agent ===

Initializing test environment...

Loading SOSA vocabulary:
Parsed JSON-LD graph with 55 entries
Added 345 triples from vocabulary http://www.w3.org/ns/sosa/
Successfully loaded SOSA vocabulary with 345 triples
Added test sensor instance

Testing simple query: What vocabularies are available in the system?

=== Agent Response ===

Message(id='msg_01Dj2MS66G1YyJPpS8cgiDDu', content=[TextBlock(citations=None, text="Currently, there is one vocabulary loaded in the system:\n\n- **SOSA Vocabulary**\n  - URI: http://www.w3.org/ns/sosa/\n  - Description: SOSA vocabulary\n  - Last Updated: 1744726226.141836 (Unix timestamp)\n\nSOSA stands for Sensor, Observation, Sample, and Actuator, which is part of the Semantic Sensor Network (SSN) ontology. This vocabulary is commonly used for describing sensors, their observations, the procedures used, the features of interest, the observed properties, and actuators.\n\nIf you'd like to:\n1. Explore the structure

(<cogitarelink.memory.SemanticMemory>,
 <cogitarelink.retriever.LODNavigator>)

### Complex Mutli-vocabulary Agent

In [ ]:
#| export
def test_semantic_agent_multivocab():
    """Test the semantic agent with cleaner output and fixed response handling"""
    print("\n=== Clean Semantic Agent Test (Fixed) ===\n")
    
    # Setup
    cache_dir = Path("./cache")
    memory = SemanticMemory(cache_dir=cache_dir)
    navigator = LODNavigator()
    
    print("Initializing test environment...")
    
    # Define vocabulary URIs properly
    vocab_uris = {
        "sosa": "http://www.w3.org/ns/sosa/",
        "prov": "http://www.w3.org/ns/prov#",
        "time": "http://www.w3.org/2006/time#",
        "geo": "http://www.opengis.net/ont/geosparql#"
    }
    
    # 1. Load vocabularies with summarized output
    print("\n1. Loading vocabularies:")
    
    for vocab_name, vocab_uri in vocab_uris.items():
        print(f"  Loading {vocab_name.upper()}...", end="")
        success, _ = memory.ensure_standard_vocabulary(vocab_name, navigator)
        
        # Count triples for this vocabulary
        vocab_triples = 0
        for s, p, o in memory.default_graph:
            if str(s).startswith(vocab_uri):
                vocab_triples += 1
                
        print(f" {'✓' if success else '✗'} ({vocab_triples} triples)")
    
    # Show total triples
    total_triples = len(list(memory.default_graph))
    print(f"\nTotal triples in graph: {total_triples}")
    
    # 2. Add instance data with simplified context
    print("\n2. Adding instance data:")
    
    # Create a simplified context
    simplified_context = {
        "@context": {
            "sosa": "http://www.w3.org/ns/sosa/",
            "prov": "http://www.w3.org/ns/prov#",
            "time": "http://www.w3.org/2006/time#",
            "geo": "http://www.opengis.net/ont/geosparql#",
            "Sensor": "sosa:Sensor",
            "Observation": "sosa:Observation",
            "Feature": "geo:Feature",
            "hasGeometry": "geo:hasGeometry",
            "asWKT": "geo:asWKT",
            "madeBySensor": "sosa:madeBySensor",
            "resultTime": "sosa:resultTime",
            "observedProperty": "sosa:observedProperty",
            "hasResult": "sosa:hasResult",
            "generatedAtTime": "prov:generatedAtTime",
            "wasAttributedTo": "prov:wasAttributedTo"
        }
    }
    
    # Register the context
    memory.register_resource(
        uri="http://example.org/contexts/test-context",
        data=simplified_context,
        resource_type="context",
        source_name="test",
        description="Test context"
    )
    
    # Add instances
    instances = [
        # Weather Station
        {
            "@context": simplified_context["@context"],
            "@id": "http://example.org/station/wx1",
            "@type": "Feature",
            "hasGeometry": {
                "@id": "http://example.org/geometry/wx1",
                "@type": "geo:Geometry",
                "asWKT": "POINT(-73.9857 40.7484)"  # New York City coordinates
            }
        },
        # Temperature Sensor
        {
            "@context": simplified_context["@context"],
            "@id": "http://example.org/sensor/temp1",
            "@type": "Sensor",
            "wasAttributedTo": {"@id": "http://example.org/station/wx1"}
        },
        # Humidity Sensor
        {
            "@context": simplified_context["@context"],
            "@id": "http://example.org/sensor/humidity1",
            "@type": "Sensor",
            "wasAttributedTo": {"@id": "http://example.org/station/wx1"}
        },
        # Temperature Observation
        {
            "@context": simplified_context["@context"],
            "@id": "http://example.org/observation/temp-2023-06-15T14:30:00Z",
            "@type": "Observation",
            "madeBySensor": {"@id": "http://example.org/sensor/temp1"},
            "observedProperty": "temperature",
            "hasResult": "25.4",
            "resultTime": "2023-06-15T14:30:00Z",
            "generatedAtTime": "2023-06-15T14:30:05Z"
        }
    ]
    
    for i, instance in enumerate(instances):
        memory.add_jsonld(instance)
        print(f"  Added {instance['@type']} instance: {instance['@id']}")
    
    # 3. Test queries with pretty-printed responses
    print("\n3. Testing queries:")
    
    # Create a simplified agent function
    def run_query(query):
        """Run a query and pretty print the response"""
        print(f"\nQuery: {query}")
        
        # Create simplified tools
        def get_entities() -> list:
            """List all entities in memory"""
            return list_entities(memory)
        
        def get_entity_details(entity_id: str) -> dict:
            """Get details about a specific entity"""
            return get_entity(memory, entity_id)
        
        def get_entity_relationships(entity_id: str) -> dict:
            """Find relationships for an entity"""
            return find_entity_relationships(memory, entity_id)
        
        # Create chat with tools
        tools = [get_entities, get_entity_details, get_entity_relationships]
        chat = Chat(model="claude-3-7-sonnet-20250219", tools=tools)
        
        prompt = f"""
        You are a Semantic Knowledge Agent that helps users explore semantic data.
        
        You have access to these tools:
        - get_entities: List all entities in memory
        - get_entity_details: Get details about a specific entity
        - get_entity_relationships: Find relationships for an entity
        
        Answer this question in a clear, concise way: {query}
        """
        
        # Use toolloop but don't print the trace
        response = chat.toolloop(prompt)
        
        # Pretty print the response
        from textwrap import fill
        
        print("\nResponse:")
        print("=" * 80)
        
        # Safely extract text from the response (handle both TextBlock and ToolUseBlock)
        if hasattr(response, 'content') and response.content:
            # Find the first TextBlock in the content
            text_blocks = [block for block in response.content if hasattr(block, 'text')]
            if text_blocks:
                response_text = text_blocks[0].text
                wrapped_text = fill(response_text, width=78)
                print(wrapped_text)
            else:
                # If no TextBlock found, just print the raw content
                print(f"No text content found. Raw content: {response.content}")
        else:
            print(f"No content found in response: {response}")
            
        print("=" * 80)
        
        return response
    
    # Run some test queries
    queries = [
        "What sensors are in the system?",
        "What observations do we have and what sensors made them?",
        "Where is the weather station located?"
    ]
    
    responses = []
    for query in queries:
        response = run_query(query)
        responses.append(response)
    
    return memory, navigator, responses

In [ ]:
mem, nav, responses = test_semantic_agent_multivocab()


=== Clean Semantic Agent Test (Fixed) ===

Initializing test environment...

1. Loading vocabularies:
  Loading SOSA...Parsed JSON-LD graph with 55 entries
Added 345 triples from vocabulary http://www.w3.org/ns/sosa/
Successfully loaded SOSA vocabulary with 345 triples
 ✓ (326 triples)
  Loading PROV...Parsed JSON-LD graph with 207 entries
Added 1664 triples from vocabulary http://www.w3.org/ns/prov#
Successfully loaded PROV vocabulary with 1664 triples
 ✓ (1426 triples)
  Loading TIME...Parsed JSON-LD graph with 157 entries
Added 1296 triples from vocabulary http://www.w3.org/2006/time#
Successfully loaded TIME vocabulary with 1296 triples
 ✓ (1078 triples)
  Loading GEO...Parsed JSON-LD graph with 78 entries
Added 804 triples from vocabulary http://www.opengis.net/ont/geosparql#
Successfully loaded GEO vocabulary with 804 triples
 ✓ (727 triples)

Total triples in graph: 4108

2. Adding instance data:
  Added Feature instance: http://example.org/station/wx1
  Added Sensor instance: 

## Older example of claude tools for Manufacturing domain.

In [ ]:
# Define simple data for our tools to use directly
CONTEXT_DATA = {
    "manufacturing": {
        "vocabulary": "http://example.org/manufacturing/",
        "key_terms": ["components", "assembledAt", "qualityScore"],
        "scoped_contexts": ["components"],
        "parent": None
    },
    "components": {
        "vocabulary": "http://example.org/components/",
        "key_terms": ["material", "supplier"],
        "scoped_contexts": [],
        "parent": "manufacturing"
    },
    "supplier": {
        "vocabulary": "http://example.org/supplier/",
        "key_terms": ["certification", "contactPerson"],
        "scoped_contexts": ["certification"],
        "parent": None
    },
    "certification": {
        "vocabulary": "http://example.org/certification/",
        "key_terms": ["type", "validUntil"],
        "scoped_contexts": [],
        "parent": "supplier"
    }
}

# Define self-contained functions that don't rely on external objects
def get_context_list() -> list:
    """Get a list of all available contexts.
    
    Returns:
        List of context IDs
    """
    return list(CONTEXT_DATA.keys())

def get_context_details(context_id: str) -> dict:
    """Get details about a specific context.
    
    Args:
        context_id: ID of the context to retrieve
        
    Returns:
        Dictionary with context details
    """
    if context_id in CONTEXT_DATA:
        return CONTEXT_DATA[context_id]
    return {"error": "Context not found"}

def find_relevant_contexts(query: str) -> list:
    """Find contexts relevant to a query.
    
    Args:
        query: Query text to analyze
        
    Returns:
        List of relevant context IDs with scores
    """
    # Simple keyword matching
    relevance = {
        "manufacturing": 0,
        "components": 0,
        "supplier": 0,
        "certification": 0
    }
    
    terms = query.lower().split()
    
    if any(word in terms for word in ["manufacturing", "product", "production"]):
        relevance["manufacturing"] += 2
    
    if any(word in terms for word in ["component", "part", "material"]):
        relevance["components"] += 2
        relevance["manufacturing"] += 1
    
    if any(word in terms for word in ["supplier", "vendor", "provide"]):
        relevance["supplier"] += 2
    
    if any(word in terms for word in ["certification", "quality", "standard"]):
        relevance["certification"] += 2
        relevance["supplier"] += 1
    
    # Return contexts with relevance > 0, sorted by relevance
    return [{"id": k, "score": v} for k, v in 
            sorted(relevance.items(), key=lambda x: x[1], reverse=True) 
            if v > 0]

def get_reasoning_hints() -> list:
    """Get hints about how to reason with these contexts.
    
    Returns:
        List of reasoning hints
    """
    return [
        "Properties using vocabulary http://example.org/manufacturing/ follow that vocabulary's semantics.",
        "The term 'components' has its own scoped context within the manufacturing context.",
        "Properties using vocabulary http://example.org/components/ follow that vocabulary's semantics.",
        "Properties using vocabulary http://example.org/supplier/ follow that vocabulary's semantics.",
        "The term 'certification' has its own scoped context within the supplier context.",
        "Properties using vocabulary http://example.org/certification/ follow that vocabulary's semantics."
    ]

def think(thought: str) -> str:
    """Use this tool to think about the context information and its implications.
    
    Args:
        thought: The thought to record
        
    Returns:
        The same thought, unchanged
    """
    return thought

def test_self_contained_tools():
    """Test self-contained context tools with Claude"""
    model = models[1]  # claude-3-7-sonnet
    
    print(f"Using model: {model}")
    
    # Create a chat with the self-contained tools
    chat = Chat(model, tools=[
        get_context_list,
        get_context_details,
        find_relevant_contexts,
        get_reasoning_hints,
        think
    ])
    
    # Test query
    prompt = """
    I want to understand how component materials are represented in the manufacturing context.
    Use the available tools to analyze which contexts are relevant and how they relate to each other.
    """
    
    print("\nSending prompt to Claude...")
    
    try:
        response = chat.toolloop(prompt)
        print(f"\nClaude's response:\n{response}")
        return response
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"

In [ ]:
test_self_contained_tools()

Using model: claude-3-7-sonnet-20250219

Sending prompt to Claude...

Claude's response:
Message(id='msg_014o4SMtqKuXqcNkV3FN7gox', content=[TextBlock(citations=None, text='Based on the information gathered, here\'s how component materials are represented in the manufacturing context:\n\n1. **Hierarchical Structure**:\n   - The "manufacturing" context is the parent context\n   - The "components" context is a scoped context within manufacturing\n   - Materials are specifically represented within the components context\n\n2. **Vocabulary and Semantics**:\n   - Manufacturing uses the vocabulary: http://example.org/manufacturing/\n   - Components uses the vocabulary: http://example.org/components/\n   - Each follows its own semantic rules according to its vocabulary\n\n3. **Key Terms for Component Materials**:\n   - Within the components context, "material" is a key term\n   - "supplier" is also a key term in the components context, suggesting materials are associated with their suppliers\

Based on the information gathered, here's how component materials are represented in the manufacturing context:

1. **Hierarchical Structure**:
   - The "manufacturing" context is the parent context
   - The "components" context is a scoped context within manufacturing
   - Materials are specifically represented within the components context

2. **Vocabulary and Semantics**:
   - Manufacturing uses the vocabulary: http://example.org/manufacturing/
   - Components uses the vocabulary: http://example.org/components/
   - Each follows its own semantic rules according to its vocabulary

3. **Key Terms for Component Materials**:
   - Within the components context, "material" is a key term
   - "supplier" is also a key term in the components context, suggesting materials are associated with their suppliers

4. **Relationship Between Contexts**:
   - The manufacturing context includes "components" as a key term and as a scoped context
   - This indicates that components (and their materials) are integral to the manufacturing process
   - The "assembledAt" and "qualityScore" terms in manufacturing likely relate to how components and their materials are used and evaluated

This representation allows for detailed tracking of component materials within the manufacturing process, with clear relationships between materials, their suppliers, and how they're used in manufacturing.

<details>

- id: `msg_014o4SMtqKuXqcNkV3FN7gox`
- content: `[{'citations': None, 'text': 'Based on the information gathered, here\'s how component materials are represented in the manufacturing context:\n\n1. **Hierarchical Structure**:\n   - The "manufacturing" context is the parent context\n   - The "components" context is a scoped context within manufacturing\n   - Materials are specifically represented within the components context\n\n2. **Vocabulary and Semantics**:\n   - Manufacturing uses the vocabulary: http://example.org/manufacturing/\n   - Components uses the vocabulary: http://example.org/components/\n   - Each follows its own semantic rules according to its vocabulary\n\n3. **Key Terms for Component Materials**:\n   - Within the components context, "material" is a key term\n   - "supplier" is also a key term in the components context, suggesting materials are associated with their suppliers\n\n4. **Relationship Between Contexts**:\n   - The manufacturing context includes "components" as a key term and as a scoped context\n   - This indicates that components (and their materials) are integral to the manufacturing process\n   - The "assembledAt" and "qualityScore" terms in manufacturing likely relate to how components and their materials are used and evaluated\n\nThis representation allows for detailed tracking of component materials within the manufacturing process, with clear relationships between materials, their suppliers, and how they\'re used in manufacturing.', 'type': 'text'}]`
- model: `claude-3-7-sonnet-20250219`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1674, 'output_tokens': 287}`

</details>

In [ ]:
#| export
# ... existing imports ...
from fastcore.basics import patch # Add patch import
import httpx # Needed for SemanticMemory client init if not implicitly imported
from datetime import datetime # Needed for _load_known_contexts if used there (though not in this version)
from rdflib import URIRef, Literal, Namespace, RDF, RDFS, Dataset # Needed for SemanticMemory internals

# Add this cell
#| export
@patch
def _load_known_contexts(self:SemanticMemory):
    """Load and register common/known contexts (patched for testing)."""
    print("Pre-loading known contexts (patched)...")

    # Define contexts relevant to supply chain and potentially Wikidata
    contexts_to_load = {
        "manufacturing": {
            "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/manufacturing/",
                "components": {
                    "@container": "@list",
                    "@context": {
                        "@vocab": "http://example.org/components/",
                        "material": "materialType",
                        "supplier": {"@type": "@id"}
                    }
                },
                "assembledAt": "productionFacility",
                "qualityScore": {"@type": "http://www.w3.org/2001/XMLSchema#decimal"}
            }
        },
        "logistics": {
             "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/logistics/",
                "shipment": {
                    "@context": {
                        "@vocab": "http://example.org/shipping/",
                        "status": "currentStatus",
                        "location": "currentLocation",
                        "estimatedArrival": {"@type": "http://www.w3.org/2001/XMLSchema#dateTime"}
                    }
                },
                "trackingNumber": "shipmentIdentifier"
            }
        },
        "retail": {
            "@context": {
                "@version": 1.1,
                "@vocab": "http://schema.org/",
                "inventory": {
                    "@context": {
                        "@vocab": "http://example.org/inventory/",
                        "level": "stockLevel",
                        "reorderPoint": "minimumStockLevel"
                    }
                },
                "price": {"@type": "http://schema.org/PriceSpecification"},
                "category": "category"
            }
        },
        "supplier": {
             "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/supplier/",
                "certification": {
                    "@context": {
                        "@vocab": "http://example.org/certification/",
                        "type": "certificationType",
                        "validUntil": {"@type": "http://www.w3.org/2001/XMLSchema#date"}
                    }
                },
                "contactPerson": "representativeName"
            }
        },
         "wikidata_basic": { # Add this for the Wikidata part
            "@context": {
                "wd": "http://www.wikidata.org/entity/",
                "wdt": "http://www.wikidata.org/prop/direct/",
                "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
                "schema": "http://schema.org/",
                "label": "rdfs:label",
                "description": "schema:description",
                "P31": "wdt:P31", # instance of
                "P279": "wdt:P279", # subclass of
            }
        }
    }

    for name, context_data in contexts_to_load.items():
        try:
            # Use _register_contexts to add them
            context_id = self._register_contexts(context_data)
            if context_id:
                self.context_registry[context_id]["source"] = f"preloaded:{name}"
                print(f"  Successfully pre-loaded context '{name}' with ID: {context_id}")
            else:
                 print(f"  Warning: Failed to get ID for preloaded context '{name}'")
        except Exception as e:
            print(f"  Warning: Failed to pre-load context '{name}': {e}")

    print("Finished pre-loading known contexts (patched).")

# ... rest of the notebook cells ...

In [ ]:
#| export
def extract_context_data(memory: SemanticMemory) -> dict:
    """Extract context data from SemanticMemory into simple data structures.
    
    Args:
        memory: SemanticMemory instance
        
    Returns:
        Dictionary of context data with simple types
    """
    contexts = memory.list_contexts()
    context_data = {}
    
    for context_id, info in contexts.items():
        try:
            formatted = memory.format_context_for_llm(context_id)
            context_data[context_id] = {
                "id": context_id,
                "vocabulary": formatted.get("vocabulary_base", ""),
                "key_terms": list(formatted.get("key_terms", {}).keys()),
                "parent": info.get("parent"),
                "has_scoped_contexts": info.get("has_scoped_contexts", False),
                "scoped_terms": info.get("scoped_context_terms", [])
            }
        except Exception as e:
            # Handle any errors gracefully
            print(f"Warning: Could not extract data for context {context_id}: {str(e)}")
            context_data[context_id] = {
                "id": context_id,
                "error": str(e)
            }
    
    return context_data

In [ ]:
#| export
def create_context_tools(memory: SemanticMemory) -> dict:
    """Create tool functions for working with context data.
    
    Args:
        memory: SemanticMemory instance
        
    Returns:
        Dictionary of tool functions
    """
    # Extract data first
    context_data = extract_context_data(memory)
    
    # Generate reasoning hints
    reasoning_hints = []
    for ctx_id, ctx in context_data.items():
        if "error" in ctx:
            continue
            
        reasoning_hints.append(f"Properties using vocabulary {ctx['vocabulary']} follow that vocabulary's semantics.")
        if ctx['has_scoped_contexts'] and ctx['scoped_terms']:
            for term in ctx['scoped_terms']:
                child_contexts = [c for c, info in context_data.items() 
                                 if "parent" in info and info['parent'] == ctx_id]
                for child in child_contexts:
                    reasoning_hints.append(f"The term '{term}' has its own scoped context with ID {child}.")
    
    # Create standalone tool functions
    def get_context_list() -> list:
        """Get a list of all available contexts.
        
        Returns:
            List of context IDs
        """
        return list(context_data.keys())
    
    def get_context_details(context_id: str) -> dict:
        """Get details about a specific context.
        
        Args:
            context_id: ID of the context to retrieve
            
        Returns:
            Dictionary with context details
        """
        if context_id in context_data:
            return context_data[context_id]
        return {"error": "Context not found"}
    
    def find_relevant_contexts(query: str) -> list:
        """Find contexts relevant to a query.
        
        Args:
            query: Query text to analyze
            
        Returns:
            List of relevant context IDs with scores
        """
        terms = query.lower().split()
        results = []
        
        for ctx_id, ctx in context_data.items():
            if "error" in ctx:
                continue
                
            score = 0
            # Match against vocabulary
            if any(term in ctx['vocabulary'].lower() for term in terms):
                score += 1
            # Match against key terms
            for key_term in ctx['key_terms']:
                if any(term in key_term.lower() for term in terms):
                    score += 2
            
            if score > 0:
                results.append({"id": ctx_id, "score": score})
        
        # Sort by score descending
        return sorted(results, key=lambda x: x["score"], reverse=True)
    
    def get_reasoning_hints() -> list:
        """Get hints about how to reason with these contexts.
        
        Returns:
            List of reasoning hints
        """
        return reasoning_hints
    
    def think(thought: str) -> str:
        """Use this tool to think about the context information and its implications.
        
        Args:
            thought: The thought to record
            
        Returns:
            The same thought, unchanged
        """
        return thought
    
    return {
        "get_context_list": get_context_list,
        "get_context_details": get_context_details,
        "find_relevant_contexts": find_relevant_contexts,
        "get_reasoning_hints": get_reasoning_hints,
        "think": think
    }

In [ ]:
#| export
def create_context_aware_chat(memory: SemanticMemory, model: str = "claude-3-7-sonnet-20250219") -> Chat:
    """Create a Claude chat with context tools.
    
    Args:
        memory: SemanticMemory instance
        model: Claude model to use
        
    Returns:
        Chat instance with context tools
    """
    # Create tool functions
    tools = create_context_tools(memory)
    
    # Create chat with all tools
    chat = Chat(model, tools=list(tools.values()))
    
    return chat

In [ ]:
#| export
def query_memory_with_claude(memory: SemanticMemory, query: str, model: str = "claude-3-7-sonnet-20250219"):
    """Query the memory using Claude with context tools.
    
    Args:
        memory: SemanticMemory instance
        query: Query text
        model: Claude model to use
        
    Returns:
        Claude's response
    """
    chat = create_context_aware_chat(memory, model)
    
    prompt = f"""
    You are an AI assistant with access to a semantic memory system that uses JSON-LD contexts.
    
    Your task is to:
    1. Identify which contexts are relevant to this query
    2. Understand how these contexts relate to each other
    3. Answer the query using information from these contexts
    
    Query: {query}
    """
    
    return chat.toolloop(prompt)

In [ ]:
# Example usage
def test_memorytools():
    """Test the memorytools module with a sample memory"""
    # Create a memory instance
    memory = SemanticMemory()
    
    # Add manufacturing context
    manufacturing_context = {
        "@version": 1.1,
        "@vocab": "http://example.org/manufacturing/",
        "components": {
            "@container": "@list",
            "@context": {
                "@vocab": "http://example.org/components/",
                "material": "materialType",
                "supplier": {"@type": "@id"}
            }
        }
    }
    
    # Add a product with this context
    product = {
        "@context": manufacturing_context,
        "@id": "http://example.org/product/x1",
        "productName": "Widget X1",
        "components": [
            {
                "name": "Frame",
                "material": "Aluminum"
            }
        ]
    }
    
    memory.add_jsonld(product)
    
    # Test extraction
    context_data = extract_context_data(memory)
    print("Extracted context data:")
    for ctx_id, ctx in context_data.items():
        print(f"  {ctx_id}: {ctx['vocabulary']}")
    
    # Test creating tools
    tools = create_context_tools(memory)
    print(f"\nCreated {len(tools)} tools")
    
    # Test a query
    query = "What materials are used in components?"
    print(f"\nQuery: {query}")
    
    try:
        response = query_memory_with_claude(memory, query)
        print(f"Response: {response}")
    except Exception as e:
        print(f"Error during query: {str(e)}")
    
    return memory

# Uncomment to run the test
# test_memorytools()

In [ ]:
test_memorytools()

Extracted context data:
  context:e99496384e7f9e645d89955bfe77dbca: http://example.org/manufacturing/
  context:0b128f19e60579c2e428fb6847eb1d16: http://example.org/components/

Created 5 tools

Query: What materials are used in components?
Response: Message(id='msg_01QYH1PchUPFsNCu98bYYjPa', content=[TextBlock(citations=None, text="Based on my analysis of the available contexts, I can answer your query about what materials are used in components:\n\n### Answer to Query: What materials are used in components?\n\nBased on the semantic memory system, there are specific materials associated with components in a manufacturing context. The information is organized in two related contexts:\n\n1. A parent context related to manufacturing (context:e99496384e7f9e645d89955bfe77dbca) that establishes the overall framework for components.\n\n2. A more specific context for components (context:0b128f19e60579c2e428fb6847eb1d16) that contains information about materials and suppliers.\n\nWhile the exa

<cogitarelink.memory.SemanticMemory>

In [ ]:
import dspy # Add dspy import

# Configure DSPy LM for this notebook's context
try:
    # Use the same model as in 01_retriever.ipynb
    lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
    dspy.configure(lm=lm)
    print("DSPy configured successfully.")
except Exception as e:
    print(f"Error configuring DSPy: {e}")
    # Optionally raise the error or handle it if configuration is critical
    # raise e

DSPy configured successfully.


In [ ]:
#| export
# ... existing imports ...
from cogitarelink.retriever import LODNavigator # Import the retriever

#| export
def test_memorytools_with_wikidata():
    """Test memorytools using data retrieved from Wikidata and supply chain examples."""
    print("\n=== Testing Memory Tools with Wikidata & Supply Chain Data ===\n")

    # --- Setup Phase ---
    # 1. Create SemanticMemory and Pre-load Contexts
    memory = SemanticMemory()
    try:
        memory._load_known_contexts() # Call the patched pre-loading method
    except AttributeError:
        print("Error: _load_known_contexts patch might not have been applied correctly.")
        return None
    except Exception as e:
        print(f"Error during context pre-loading: {e}")
        return None

    # 2. Add Supply Chain Data (copied from 00_memory.ipynb test_supply_chain_example)
    print("\nAdding Supply Chain data to SemanticMemory...")
    try:
        # Manufacturing context data
        manufacturing_context = {
            "@version": 1.1, "@vocab": "http://example.org/manufacturing/",
            "components": {"@container": "@list", "@context": {"@vocab": "http://example.org/components/", "material": "materialType", "supplier": {"@type": "@id"}}},
            "assembledAt": "productionFacility", "qualityScore": {"@type": "http://www.w3.org/2001/XMLSchema#decimal"}
        }
        smartphone_product = {
            "@context": manufacturing_context, "@id": "http://example.org/product/smartphone-x200", "@type": "Product",
            "productName": "Smartphone X200", "productionDate": "2023-10-15", "batchNumber": "SX200-2023-10-B42",
            "assembledAt": "Shenzhen Factory 7", "qualityScore": "9.8",
            "components": [
                {"@id": "http://example.org/component/screen-amoled", "name": "AMOLED Display", "material": "Glass-Composite", "partNumber": "SCR-X200-AM", "supplier": "http://example.org/supplier/displaytech"},
                {"@id": "http://example.org/component/battery-lithium", "name": "Lithium Battery Pack", "material": "Lithium-Ion", "partNumber": "BAT-5000mAh", "supplier": "http://example.org/supplier/powerplus"},
                {"@id": "http://example.org/component/processor-a12", "name": "A12 Processor", "material": "Silicon", "partNumber": "CPU-A12-3.1", "supplier": "http://example.org/supplier/chipworks"}
            ]
        }
        memory.add_jsonld(smartphone_product) # Use regular add_jsonld as it has context

        # Logistics context data
        logistics_context = {
            "@version": 1.1, "@vocab": "http://example.org/logistics/",
            "shipment": {"@context": {"@vocab": "http://example.org/shipping/", "status": "currentStatus", "location": "currentLocation", "estimatedArrival": {"@type": "http://www.w3.org/2001/XMLSchema#dateTime"}}},
            "trackingNumber": "shipmentIdentifier"
        }
        smartphone_shipment = {
            "@context": logistics_context, "@id": "http://example.org/shipment/sx200-batch-42", "@type": "Shipment",
            "shipmentDate": "2023-10-20", "contents": "Smartphone X200 - 1000 units", "origin": "Shenzhen Factory 7",
            "destination": "North American Distribution Center", "trackingNumber": "SHP-2023-10-42-NAM", "carrier": "Global Logistics Partners",
            "shipment": {"status": "In Transit", "location": "Pacific Ocean", "estimatedArrival": "2023-11-05T08:00:00Z", "mode": "Sea Freight", "container": "GLPC-98765"}
        }
        memory.add_jsonld(smartphone_shipment)

        # Add more supply chain data if needed (retail, supplier) following the pattern...
        print("Supply Chain data added successfully.")

    except Exception as e:
        print(f"Error adding Supply Chain data to memory: {e}")
        import traceback
        traceback.print_exc()
        # Decide if test should continue

    # 3. Fetch and Add Wikidata Data
    print("\nFetching and adding Wikidata data...")
    navigator = LODNavigator()
    wikidata_uri = "http://www.wikidata.org/entity/Q42" # Douglas Adams
    print(f"Fetching data for {wikidata_uri}...")
    retrieval_result = navigator.navigate(wikidata_uri)

    if not retrieval_result.get("success"):
        print(f"Failed to retrieve Wikidata data: {retrieval_result.get('error')}")
        # Decide if test should continue
    else:
        wikidata_jsonld = retrieval_result.get("json_ld")
        print(f"Successfully retrieved Wikidata data.")
        print("Adding Wikidata data to SemanticMemory...")
        try:
            # Use add_jsonld_with_graph if the result might be a graph structure
            if isinstance(wikidata_jsonld, dict) and '@graph' in wikidata_jsonld:
                 memory.add_jsonld_with_graph(wikidata_jsonld)
            elif wikidata_jsonld: # Ensure it's not None
                 memory.add_jsonld(wikidata_jsonld)
            else:
                 print("Warning: Retrieved Wikidata data was empty or None.")
            print("Wikidata data added successfully (or skipped if empty).")
        except Exception as e:
            print(f"Error adding Wikidata data to memory: {e}")
            import traceback
            traceback.print_exc()
            # Decide if test should continue

    # --- Query Phase ---
    # 4. Query with memorytools
    # Query focusing on Wikidata data, but context tools now know about supply chain contexts too
    query = "What is Douglas Adams (Q42) known for, based on the data? Also, list the available contexts."
    print(f"\nQuerying memory with Claude: '{query}'")

    try:
        response = query_memory_with_claude(memory, query)
        print(f"\nClaude's response:\n{response}")
        # Add assertions here if needed, e.g., check if response mentions Adams or contexts
        assert response is not None, "Claude response should not be None"
        # assert "Douglas Adams" in str(response), "Response should mention Douglas Adams" # Example assertion
        # assert "manufacturing" in str(response), "Response should mention manufacturing context" # Example assertion
        return response
    except Exception as e:
        print(f"Error during query_memory_with_claude: {e}")
        import traceback
        traceback.print_exc()
        return None

# ... rest of the notebook ...

In [ ]:
test_memorytools_with_wikidata()


=== Testing Memory Tools with Wikidata & Supply Chain Data ===

Pre-loading known contexts (patched)...
  Successfully pre-loaded context 'manufacturing' with ID: context:20de30c246474eabdebd3ab17cf788d2
  Successfully pre-loaded context 'logistics' with ID: context:eba5b00877103f96b9c4c711cb21542d
  Successfully pre-loaded context 'retail' with ID: context:6149af7dc8d667bf303e98259699be18
  Successfully pre-loaded context 'supplier' with ID: context:2f8b290d50bc68c6127a921c11a848a3
  Successfully pre-loaded context 'wikidata_basic' with ID: context:e4b3a08e3d2ffc258a12375ab3c4f90c
Finished pre-loading known contexts (patched).

Adding Supply Chain data to SemanticMemory...
Supply Chain data added successfully.

Fetching and adding Wikidata data...
Fetching data for http://www.wikidata.org/entity/Q42...
Successfully retrieved Wikidata data.
Adding Wikidata data to SemanticMemory...
Wikidata data added successfully (or skipped if empty).

Querying memory with Claude: 'What is Douglas A

ToolUseBlock(id='toolu_01P26zPGaNTxrK4YTHrxdQBh', input={'context_id': 'context:2f8b290d50bc68c6127a921c11a848a3'}, name='get_context_details', type='tool_use')

<details>

- id: `msg_01C1RUaYTXta6VLnbNnEBxhL`
- content: `[{'id': 'toolu_01P26zPGaNTxrK4YTHrxdQBh', 'input': {'context_id': 'context:2f8b290d50bc68c6127a921c11a848a3'}, 'name': 'get_context_details', 'type': 'tool_use'}]`
- model: `claude-3-7-sonnet-20250219`
- role: `assistant`
- stop_reason: `tool_use`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2564, 'output_tokens': 79}`

</details>

In [ ]:
#| export
@patch
def summarize_context(self:SemanticMemory, context_id):
    """Generate a human-readable summary of a context structure"""
    if context_id not in self.context_registry:
        return {"error": f"Context {context_id} not found"}
    
    context = self.context_registry[context_id]["context"]
    formatted = self.format_context_for_llm(context_id)
    
    summary = {
        "id": context_id,
        "vocabulary_base": formatted.get("vocabulary_base", "No base vocabulary"),
        "key_terms": {},
        "scoped_contexts": {},
        "parent_context": self.context_registry[context_id].get("parent")
    }
    
    # Extract and describe key terms
    for term, details in formatted.get("key_terms", {}).items():
        summary["key_terms"][term] = {
            "maps_to": details.get("mapped_to", "Unknown"),
            "has_scoped_context": details.get("has_scoped_context", False),
            "coercion_type": details.get("type_coercion")
        }
    
    # Identify scoped contexts
    for term, scoped_id in self.context_registry[context_id].get("scoped_contexts", {}).items():
        summary["scoped_contexts"][term] = scoped_id
    
    return summary

In [ ]:
#| export
def format_response_rich(response, format_type="markdown"):
    """Format Claude's response with rich formatting options
    
    Args:
        response: Claude response object
        format_type: Type of formatting to apply (markdown, plain, highlight)
    
    Returns:
        Formatted string
    """
    # Extract the text content
    text = ""
    if hasattr(response, 'content'):
        for block in response.content:
            if hasattr(block, 'text'):
                text = block.text
                break
    else:
        text = str(response)
    
    if format_type == "plain":
        # Just return the plain text
        return text
    
    elif format_type == "highlight":
        # Highlight key information like entities, numbers, etc.
        import re
        # Highlight URIs
        text = re.sub(r'`(http[^`]+)`', r'\033[36m\1\033[0m', text)
        # Highlight component names
        text = re.sub(r'\*\*(.*?)\*\*', r'\033[1m\1\033[0m', text)
        # Highlight numbers
        text = re.sub(r'(\d+\.?\d*)', r'\033[33m\1\033[0m', text)
        return text
    
    # Default is markdown - no additional processing needed
    return text

In [ ]:
#| export
@patch
def find_entities(self:SemanticMemory, query_params):
    """Find entities based on type, property values, or keywords"""
    results = []
    
    if "type" in query_params:
        entities = self.query_by_type(query_params["type"])
        if entities: results.extend(entities)
    
    if "property" in query_params and "value" in query_params:
        # Search for entities with matching property-value pair
        prop, val = query_params["property"], query_params["value"]
        for entity_id, entity in self.indices["by_id"].items():
            if prop in entity and any(v.get("@value") == val or v.get("@id") == val for v in entity[prop] if isinstance(v, dict)):
                if entity not in results: results.append(entity)
    
    if "keyword" in query_params:
        keyword = query_params["keyword"].lower()
        # Search in labels and descriptions
        if "by_label" in self.indices:
            for label, entities in self.indices["by_label"].items():
                if keyword in label.lower():
                    for entity in entities:
                        if entity not in results: results.append(entity)
                        
        # Also search in values
        for entity_id, entity in self.indices["by_id"].items():
            entity_str = str(entity).lower()
            if keyword in entity_str and entity not in results:
                results.append(entity)
    
    return results

In [ ]:
#| export
@patch
def get_entities_by_type(self:SemanticMemory, type_uri:str) -> list:
    """Get all entities of a specific type with their properties.
    
    Args:
        type_uri: URI of the type to retrieve
        
    Returns:
        List of entities with that type
    """
    entities = self.query_by_type(type_uri)
    
    # For each entity, retrieve it with its original context
    result = []
    for entity in entities:
        if "@id" in entity:
            entity_with_context = self.retrieve_in_context(entity["@id"])
            if entity_with_context:
                result.append(entity_with_context)
            else:
                result.append(entity)
        else:
            result.append(entity)
    
    return result

In [ ]:
#| export
def create_entity_finder_tools(memory: SemanticMemory) -> dict:
    """Create generalized tools for finding and working with entities"""
    # Get standard context tools
    tools = create_context_tools(memory)
    
    # Add entity finder tools
    def find_by_type(type_uri: str) -> list:
        """Find all entities of a specific type
        
        Args:
            type_uri: URI of the type to find
        
        Returns:
            List of matching entities
        """
        return memory.find_entities({"type": type_uri})
    
    def find_by_property(property_uri: str, value: str) -> list:
        """Find entities with a specific property value
        
        Args:
            property_uri: URI of the property to search
            value: Value to match (string or URI)
        
        Returns:
            List of matching entities
        """
        return memory.find_entities({"property": property_uri, "value": value})
    
    def find_by_keyword(keyword: str) -> list:
        """Find entities containing a keyword in their labels or values
        
        Args:
            keyword: Keyword to search for
        
        Returns:
            List of matching entities
        """
        return memory.find_entities({"keyword": keyword})
    
    def get_entity_properties(entity_id: str) -> dict:
        """Get all properties of a specific entity
        
        Args:
            entity_id: ID of the entity to retrieve
        
        Returns:
            Entity with all its properties
        """
        entity = memory.retrieve_in_context(entity_id) 
        return entity or memory.query_by_id(entity_id)
    
    def list_entity_types() -> list:
        """List all entity types in the memory system
        
        Returns:
            List of entity types with counts
        """
        return [(type_uri, len(entities)) for type_uri, entities in memory.indices["by_type"].items()]
    
    # Add all tools
    tools.update({
        "find_by_type": find_by_type,
        "find_by_property": find_by_property,
        "find_by_keyword": find_by_keyword,
        "get_entity_properties": get_entity_properties,
        "list_entity_types": list_entity_types
    })
    
    return tools

In [ ]:
#| export
def create_navigation_tools(memory: SemanticMemory) -> dict:
    """Create tools for navigating and exploring semantic memory"""
    tools = {}
    
    def list_contexts() -> dict:
        """List all available contexts with their descriptions"""
        contexts = memory.list_contexts()
        return {ctx_id: {
            "entity_count": info.get("entity_count", 0),
            "has_scoped_contexts": info.get("has_scoped_contexts", False),
            "parent": info.get("parent")
        } for ctx_id, info in contexts.items()}
    
    def explore_context(context_id: str) -> dict:
        """Explore the structure of a specific context"""
        return memory.summarize_context(context_id)
    
    def get_entity_types() -> list:
        """Get all entity types in the system with counts"""
        return [(type_uri, len(entities)) for type_uri, entities in memory.indices["by_type"].items()]
    
    def explore_entity_type(type_uri: str) -> dict:
        """Explore entities of a specific type"""
        entities = memory.query_by_type(type_uri)
        return {
            "type": type_uri,
            "count": len(entities),
            "sample_entities": [e.get("@id") for e in entities[:3] if "@id" in e],
            "common_properties": _get_common_properties(entities)
        }
    
    def explore_entity(entity_id: str) -> dict:
        """Explore a specific entity and its properties"""
        entity = memory.retrieve_in_context(entity_id) or memory.query_by_id(entity_id)
        if not entity:
            return {"error": f"Entity {entity_id} not found"}
        
        # Extract basic info
        result = {"id": entity_id, "properties": {}}
        
        # Extract property values in a more readable format
        for prop, values in entity.items():
            if prop in ["@id", "@type", "@context"]:
                continue
                
            # Format values
            formatted_values = []
            for val in values:
                if isinstance(val, dict):
                    if "@value" in val:
                        formatted_values.append({"value": val["@value"], "type": val.get("@type")})
                    elif "@id" in val:
                        formatted_values.append({"id": val["@id"]})
                    elif "@list" in val:
                        formatted_values.append({"list": val["@list"]})
                else:
                    formatted_values.append({"value": val})
            
            result["properties"][prop] = formatted_values
        
        return result
    
    def follow_relationship(entity_id: str, relationship_uri: str) -> list:
        """Follow a relationship from an entity to related entities"""
        entity = memory.query_by_id(entity_id)
        if not entity:
            return {"error": f"Entity {entity_id} not found"}
            
        # Find values for this relationship
        related_entities = []
        if relationship_uri in entity:
            for val in entity[relationship_uri]:
                if isinstance(val, dict) and "@id" in val:
                    related = memory.query_by_id(val["@id"])
                    if related:
                        related_entities.append(related)
        
        return related_entities
    
    def search_memory(query: str) -> list:
        """Search for entities matching a text query"""
        results = []
        lower_query = query.lower()
        
        # Search in labels
        if "by_label" in memory.indices:
            for label, entities in memory.indices["by_label"].items():
                if lower_query in label.lower():
                    for entity in entities:
                        if entity not in results and "@id" in entity:
                            results.append({"id": entity["@id"], "matched_on": "label", "label": label})
        
        # Search in all values
        for entity_id, entity in memory.indices["by_id"].items():
            entity_str = str(entity).lower()
            if lower_query in entity_str:
                if not any(r["id"] == entity_id for r in results):
                    results.append({"id": entity_id, "matched_on": "content"})
        
        return results
    
    def think(thought: str) -> str:
        """Use this tool to think about the semantic structure and connections"""
        return thought
    
    # Helper function to identify common properties
    def _get_common_properties(entities):
        if not entities:
            return []
            
        # Count property occurrences
        prop_counts = {}
        for entity in entities:
            for prop in entity.keys():
                if prop.startswith('@'):
                    continue
                prop_counts[prop] = prop_counts.get(prop, 0) + 1
        
        # Return properties that appear in at least half the entities
        threshold = max(1, len(entities) // 2)
        return [prop for prop, count in prop_counts.items() if count >= threshold]
    
    # Add all tools
    tools.update({
        "list_contexts": list_contexts,
        "explore_context": explore_context,
        "get_entity_types": get_entity_types,
        "explore_entity_type": explore_entity_type,
        "explore_entity": explore_entity,
        "follow_relationship": follow_relationship,
        "search_memory": search_memory,
        "think": think
    })
    
    return tools

## NBDev Export
Export the notebook cells to the library.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()